<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# FABnet IPv6 Network: Manual Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** Creates a two-node experiment spanning two FABRIC sites connected via **FABnet IPv6**, then walks you through **manually** assigning IPv6 addresses and configuring routes after the slice becomes active. This gives you full control over IP address assignment.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create FABnet IPv6 networks without automatic IP configuration
2. Query the assigned subnet and list of available IPs from each network
3. Manually assign IPv6 addresses to node interfaces using `ip_addr_add()`
4. Manually add inter-site routes using `ip_route_add()`
5. Verify the configuration using `ip addr show` and `ip route list`

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with creating basic slices -- see [Hello, FABRIC](../../hello_fabric/hello_fabric.ipynb)

**Tip -- Three configuration approaches:** FABRIC offers three ways to set up FABnet IPv6:

| Approach | Notebook | You create networks? | You assign IPs? |
|----------|----------|---------------------|-----------------|
| **Full Auto** | [full_auto](create_l3network_fabnet_ipv6_full_auto.ipynb) | No (`add_fabnet()`) | No |
| **Auto** | [auto](create_l3network_fabnet_ipv6_auto.ipynb) | Yes | No (auto mode) |
| **Manual** (this notebook) | You are here | Yes | Yes |

</div>

## Background: Manual IPv6 Configuration

FABRIC provides Layer 3 networking (FABnetv4 and FABnetv6) across every site. With **manual** configuration, the slice is submitted without any IP address setup. After the slice becomes active, you:

1. **Query** each network for its FABRIC-assigned subnet and available IPs
2. **Pick** an IP from the available list and assign it to the node's interface
3. **Add routes** so each node can reach the other site's subnet via the local gateway

This approach is useful when you need precise control over IP assignment -- for example, when integrating with external systems that expect specific addresses.


**NIC component model options:**

| Model | Speed | Type | Ports |
|-------|-------|------|-------|
| `NIC_Basic` | 100 Gbps | Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps | Dedicated Mellanox ConnectX-5 | 2 |
| `NIC_ConnectX_6` | 100 Gbps | Dedicated Mellanox ConnectX-6 | 2 |

## What We're Building

In this notebook we will create two nodes on different sites connected via FABNetv6.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We select two distinct random sites. Each site will have its own FABnet IPv6 network.

In [ ]:
# Name for this experiment slice
slice_name = 'MySlice'

# Pick two distinct random FABRIC sites
[site1, site2] = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node names
node1_name = 'Node1'
node2_name = 'Node2'

# Network names -- one per site
network1_name = 'net1'
network2_name = 'net2'

# NIC names
node1_nic_name = 'nic1'
node2_nic_name = 'nic2'

## Step 3: Build and Submit the Slice

In manual mode, we create the nodes and networks but do **not** set auto mode on interfaces. No IP addresses are configured at this stage -- we will do that after the slice is active.

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Node1 (site1) ---
node1 = slice.add_node(name=node1_name, site=site1)
# Add a basic NIC -- note: no set_mode('auto') call for manual config
iface1 = node1.add_component(model='NIC_Basic', name=node1_nic_name).get_interfaces()[0]

# --- Node2 (site2) ---
node2 = slice.add_node(name=node2_name, site=site2)
iface2 = node2.add_component(model='NIC_Basic', name=node2_nic_name).get_interfaces()[0]

# --- Networks: pass interfaces at creation time ---
net1 = slice.add_l3network(name=network1_name, interfaces=[iface1], type='IPv6')
net2 = slice.add_l3network(name=network2_name, interfaces=[iface2], type='IPv6')

# Submit the slice -- blocks until provisioning is complete (~2-5 min)
slice.submit();

---

## Step 4: Manually Configure IP Addresses

Now that the slice is active, FABRIC has assigned a subnet and gateway to each network. We need to:

1. Query the available IPs from each network
2. Assign one IP to each node's interface
3. Add routes between the two subnets

### Step 4a: Query Network Subnets

In [ ]:
# Get the network objects and their available IP addresses
network1 = slice.get_network(name=network1_name)
network1_available_ips = network1.get_available_ips()
network1.show()

network2 = slice.get_network(name=network2_name)
network2_available_ips = network2.get_available_ips()
network2.show();

### Step 4b: Configure Node1

We assign the first available IP from `network1` to Node1's interface, then add a route to reach `network2`'s subnet via `network1`'s gateway.

In [ ]:
# Get the Node1 object and its interface connected to network1
node1 = slice.get_node(name=node1_name)
node1_iface = node1.get_interface(network_name=network1_name)

# Pop the first available IP and assign it to the interface
node1_addr = network1_available_ips.pop(0)
node1_iface.ip_addr_add(addr=node1_addr, subnet=network1.get_subnet())

# Add a route: to reach network2's subnet, go via network1's gateway
node1.ip_route_add(subnet=network2.get_subnet(), gateway=network1.get_gateway())

# Verify: show the interface configuration and routing table
stdout, stderr = node1.execute(f'ip addr show {node1_iface.get_device_name()}')
stdout, stderr = node1.execute(f'ip route list')

### Step 4c: Configure Node2

Repeat the same process for Node2 using `network2`'s available IPs and adding a route back to `network1`.

In [ ]:
# Get the Node2 object and its interface connected to network2
node2 = slice.get_node(name=node2_name)
node2_iface = node2.get_interface(network_name=network2_name)

# Pop the first available IP and assign it to the interface
node2_addr = network2_available_ips.pop(0)
node2_iface.ip_addr_add(addr=node2_addr, subnet=network2.get_subnet())

# Add a route: to reach network1's subnet, go via network2's gateway
node2.ip_route_add(subnet=network1.get_subnet(), gateway=network2.get_gateway())

# Verify: show the interface configuration and routing table
stdout, stderr = node2.execute(f'ip addr show {node2_iface.get_device_name()}')
stdout, stderr = node2.execute(f'ip route list')

---

## Step 5: Run the Experiment

Now that both nodes have IPv6 addresses and routes, we verify connectivity by pinging Node2 from Node1.

In [ ]:
# Retrieve the node (useful if re-running this cell later)
node1 = slice.get_node(name=node1_name)

# Ping Node2's IPv6 address from Node1
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Step 6: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice = fablib.get_slice(name=slice_name)
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `ip_addr_add()` fails | Interface not found | Verify `network_name` matches what was used in `add_l3network()` |
| `ping` fails between nodes | Routes not configured | Run `ip route list` on both nodes to verify routes exist |
| `get_available_ips()` returns empty list | All IPs already assigned | Check if another experiment is using the same network |
| `Network unreachable` | Gateway not reachable | Verify the IP was assigned to the correct interface with `ip addr show` |
| Slice stuck in `Configuring` | Site may be busy | Try different sites by re-running `get_random_sites()` |

## FABlib API Reference

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.get_random_sites(count)` | Select distinct random sites | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |
| `slice.add_l3network(name, interfaces, type)` | Add a Layer 3 network | [add_l3network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l3network) |
| `slice.get_network(name)` | Get a network object by name | [get_network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_network) |
| `network.get_available_ips()` | List available IPs on the network | [get_available_ips](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_available_ips) |
| `network.get_subnet()` | Get the network's subnet | [get_subnet](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_subnet) |
| `network.get_gateway()` | Get the network's gateway IP | [get_gateway](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_gateway) |
| `iface.ip_addr_add(addr, subnet)` | Assign an IP address to an interface | [ip_addr_add](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.ip_addr_add) |
| `node.ip_route_add(subnet, gateway)` | Add a static route on the node | [ip_route_add](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.ip_route_add) |
| `iface.get_device_name()` | Get the Linux device name of the interface | [get_device_name](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_device_name) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **FABnet IPv6 Full Auto** | [full_auto](create_l3network_fabnet_ipv6_full_auto.ipynb) | Simplest approach -- `add_fabnet()` handles everything |
| **FABnet IPv6 Auto** | [auto](create_l3network_fabnet_ipv6_auto.ipynb) | Auto IP assignment with manual network creation |
| **FABnet IPv4 Manual** | [ipv4_manual](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_manual.ipynb) | Same pattern with IPv4 addressing |
| **FABnet IPv4 Ext** | [ipv4_ext](../create_l3network_fabnet_ipv4ext_manual/create_l3network_fabnet_ipv4ext_manual.ipynb) | IPv4 with external (public) connectivity |
| **FABnet IPv6 Ext** | [ipv6_ext](../create_l3network_fabnet_ipv6ext_manual/create_l3network_fabnet_ipv6ext_manual.ipynb) | IPv6 with external (public) connectivity |